# Smart-Grid Datasets and Dashboard

"Smart_Grid" data cleaing, preprocessing, and dashboard visualization.

In [ ]:
%pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.functions import col, sum

%pip install --upgrade pandas
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler

%pip install dash plotly

In [ ]:
# Download dataset
import requests
path = "https://raw.githubusercontent.com/huqingyuan314/DS5110-25Summer-Project/refs/heads/main/datasets/Smart%20Grid%20Real-Time%20Load%20Monitoring%20Dataset/smart_grid_dataset.csv"
req = requests.get(path)
url_content = req.content

csv_file_name = 'smart_grid_data.csv'
with open(csv_file_name, 'wb') as csv_file:
    csv_file.write(url_content)


# Start Spark
spark = SparkSession.builder.appName("Smart_Grid").getOrCreate()

# Read into PySpark DataFrame
df = spark.read.csv(csv_file_name, header=True, inferSchema=True)
df.printSchema()
df.show(5)

# Create TempView
df.createOrReplaceTempView("smart_grid_data")

In [ ]:
# Check if there is any null value. (Count nulls in each column)

df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

In [ ]:
# Query test: Number of records by year.

spark.sql("""
    SELECT 
        SUBSTRING(Timestamp, 1, 4) AS Year,
        COUNT(*) AS Num_Records
    FROM smart_grid_data
    GROUP BY Year
    ORDER BY Year
""").show()

In [ ]:
'''
=============================================================
Plotly Dash app.
Run this chunk, then go to http://127.0.0.1:8050/ in browser.
=============================================================
'''

# from werkzeug._internal import _plain_int

import dash
from dash import dcc, html, dash_table, Input, Output
import pandas as pd
import plotly.express as px

# === Load CSV ===
df = pd.read_csv("smart_grid_data.csv", parse_dates=["Timestamp"])

# === Extract date parts ===
df["Year"] = df["Timestamp"].dt.year.astype(str)
df["Month"] = df["Timestamp"].dt.strftime("%Y-%m")
df["Day"] = df["Timestamp"].dt.strftime("%Y-%m-%d")



# ======================
# === Build Dash App ===
# ======================
app = dash.Dash(__name__)
app.title = "Smart Grid Dashboard"

app.layout = html.Div([
    html.H1("Smart Grid Monitoring Dashboard"),

    html.Div([
        html.Div([
            html.Label("Year"),
            dcc.Dropdown(sorted(df["Year"].unique()), id="year-dropdown", clearable=True)
        ], style={"width": "30%", "display": "inline-block"}),

        html.Div([
            html.Label("Month"),
            dcc.Dropdown(sorted(df["Month"].unique()), id="month-dropdown", clearable=True)
        ], style={"width": "30%", "display": "inline-block", "marginLeft": "2%"}),

        html.Div([
            html.Label("Day"),
            dcc.Dropdown(sorted(df["Day"].unique()), id="day-dropdown", clearable=True)
        ], style={"width": "30%", "display": "inline-block", "marginLeft": "2%"})
    ], style={"marginBottom": "20px"}),

    # Charts
    html.Div([
        dcc.Graph(id="voltage-graph"),
        dcc.Graph(id="current-graph"),
        dcc.Graph(id="power-consumption-graph"),
        dcc.Graph(id="predicted-vs-actual-graph"),
        dcc.Graph(id="renewable-graph"),
        dcc.Graph(id="price-graph"),
        dcc.Graph(id="binary-pie-graph"),
        dcc.Graph(id="temp-humidity-scatter"),
    ], style={"display": "grid", "gridTemplateColumns": "1fr 1fr", "gap": "20px"}),

    html.H3("Filtered Data Table"),
        dash_table.DataTable(
        id="data-table",
        columns=[{"name": i, "id": i} for i in df.columns],
        page_size=10,
        style_table={"overflowX": "auto", "maxHeight": "400px", "overflowY": "auto"},
        style_cell={"textAlign": "left"}
    )
])


# === Update month options when year is selected ===
@app.callback(
    Output('month-dropdown', 'options'),
    Output('month-dropdown', 'value'),
    Input('year-dropdown', 'value')
)
def update_months(year):
    if not year:
        return [], None
    months = sorted(df[df['Year'] == year]['Month'].unique())
    return [{"label": m, "value": m} for m in months], None


# === Update day options when month is selected ===
@app.callback(
    Output('day-dropdown', 'options'),
    Output('day-dropdown', 'value'),
    Input('year-dropdown', 'value'),
    Input('month-dropdown', 'value')
)
def update_days(year, month):
    if not (year and month):
        return [], None
    days = sorted(df[
        (df['Year'] == year) & (df['Month'] == month)
    ]['Day'].unique())
    return [{"label": d, "value": d} for d in days], None

@app.callback(
    Output("voltage-graph", "figure"),
    Output("current-graph", "figure"),
    Output("power-consumption-graph", "figure"),
    Output("predicted-vs-actual-graph", "figure"),
    Output("renewable-graph", "figure"),
    Output("price-graph", "figure"),
    Output("binary-pie-graph", "figure"),
    Output("temp-humidity-scatter", "figure"),
    Output("data-table", "data"),
    Input("year-dropdown", "value"),
    Input("month-dropdown", "value"),
    Input("day-dropdown", "value"),
)


def update_dashboard(year, month, day):
    # === Filter Data ===
    dff = df.copy()
    if year:
        dff = dff[dff["Year"] == year]
    if month:
        dff = dff[dff["Month"] == month]
    if day:
        dff = dff[dff["Day"] == day]


    # === Plot 1 & 2: Voltage (V) & Current (A) Time Series ===
    fig1 = px.line(dff, x="Timestamp", y="Voltage (V)", title="Voltage Over Time")
    fig2 = px.line(dff, x="Timestamp", y="Current (A)", title="Current Over Time")

    # === Plot 3: Power Consumption Time Series ===
    fig3 = px.line(dff, x="Timestamp", y="Power Consumption (kW)", title="Power Consumption Over Time")

    # === Plot 4: Predicted vs Actual ===
    fig4 = px.line(dff, x="Timestamp", y=["Power Consumption (kW)", "Predicted Load (kW)"],
                   title="Predicted vs Actual Load")

    # === Plot 5: Renewable Power ===
    fig5 = px.line(dff, x="Timestamp", y=["Solar Power (kW)", "Wind Power (kW)"],
                   title="Renewable Power Over Time")

    # === Plot 6: Electricity Price ===
    fig6 = px.line(dff, x="Timestamp", y="Electricity Price (USD/kWh)", title="Electricity Price Over Time")

    # === Plot 7: Binary Pie Charts ===
    pie_df = dff[["Overload Condition", "Transformer Fault"]].melt()
    fig7 = px.pie(pie_df, names='value', title="Overload / Transformer Fault Distribution")

    # === Plot 8: Temp vs Humidity ===
    fig8 = px.scatter(dff, x="Temperature (°C)", y="Humidity (%)",
                      color="Electricity Price (USD/kWh)", title="Temperature vs Humidity")

    return fig1, fig2, fig3, fig4, fig5, fig6, fig7, fig8, dff.to_dict("records")

# === Program Entry ===
if __name__ == "__main__":
    app.run(debug=True)